# XGBoost

Nesse notebook nós iremos treinar um modelo $\text{XGBoost}$ em nosso dataset médico. O $\text{XGBoost}$ é um algoritmo de gradient boosting baseado em árvores de decisão, amplamente utilizado em competições de machine learning por seu excelente desempenho em dados tabulares e sua capacidade de lidar nativamente com valores faltantes.

## Importando as Bibliotecas

Primeiro vamos carregar nossas bibliotecas.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


## Carregando o Dataset

Agora vamos carregar nosso dataset.

In [2]:
df = pd.read_parquet("../../data/processed/UCMF_fitted.parquet")


In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 11705 entries, 0 to 12872
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   peso                    9587 non-null   Float64 
 1   altura                  8078 non-null   Int64   
 2   imc                     7708 non-null   Int64   
 3   idade                   10865 non-null  Float64 
 4   pulsos                  11657 non-null  category
 5   pa_sistolica            5131 non-null   Int64   
 6   pa_diastolica           5121 non-null   Int64   
 7   ppa                     10768 non-null  category
 8   patologia               11705 non-null  category
 9   b2                      11674 non-null  category
 10  sopro                   11682 non-null  category
 11  fc                      10986 non-null  Int64   
 12  hda1                    8565 non-null   category
 13  hda2                    11705 non-null  category
 14  sexo                    11701 non-null

## Seleção de Modelo

Assim como fizemos com o $\text{KNN}$ e o $\text{SVM}$, vamos selecionar o melhor modelo de $\text{XGBoost}$ otimizando os hiperparâmetros `n_estimators`, `max_depth` e `learning_rate` usando o `GridSearchCV`. Usamos `OrdinalEncoder` ao invés de `OneHotEncoder` pois modelos baseados em árvore não são sensíveis à magnitude das codificações, evitando também a explosão de dimensionalidade.

In [4]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
    cross_validate,
    train_test_split,
)

random_state = 42
cross_validator = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state,
)


def create_grid_search(X):
    numeric_columns = X.select_dtypes(include="number").columns.to_list()
    categorical_columns = X.select_dtypes(include="category").columns.to_list()

    model = XGBClassifier(
        eval_metric="logloss",
        random_state=random_state,
        n_jobs=-1,
    )
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric_features", numeric_pipeline, numeric_columns),
            ("categorical_features", categorical_pipeline, categorical_columns),
        ],
        remainder="passthrough",
    )

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    param_grid = {
        "model__n_estimators": [200, 500],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.05, 0.1],
    }

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cross_validator,
        n_jobs=-1,
    )

    return grid_search

label = LabelEncoder()

X = df.drop(["patologia"], axis="columns")
y = label.fit_transform(df["patologia"])


Agora vamos criar nosso otimizador.

In [5]:
grid_search = create_grid_search(X)

E então vamos aplicar a otimização para que ele encontre os melhores valores para os nossos hiperparâmetros.

In [6]:
grid_search.fit(X, y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.05, 0.1], 'model__max_depth': [3, 5, ...], 'model__n_estimators': [200, 500]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each 

Com a busca concluída podemos acessar os melhores parâmetros abaixo.

In [7]:
grid_search.best_params_

{'model__learning_rate': 0.05,
 'model__max_depth': 5,
 'model__n_estimators': 200}

Com isso podemos montar nossa pipeline com os melhores parâmetros encontrados.

In [8]:
best_n_estimators = grid_search.best_params_["model__n_estimators"]
best_max_depth    = grid_search.best_params_["model__max_depth"]
best_lr           = grid_search.best_params_["model__learning_rate"]

def create_pipeline(X):
    numeric_columns = X.select_dtypes(include="number").columns.to_list()
    categorical_columns = X.select_dtypes(include="category").columns.to_list()

    model = XGBClassifier(
        n_estimators=best_n_estimators,
        max_depth=best_max_depth,
        learning_rate=best_lr,
        eval_metric="logloss",
        random_state=random_state,
        n_jobs=-1,
    )
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )),
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric_features", numeric_pipeline, numeric_columns),
            ("categorical_features", categorical_pipeline, categorical_columns),
        ],
        remainder="passthrough",
    )

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    return pipeline

pipeline = create_pipeline(X)


## Treinando o Modelo

Agora vamos avaliar o desempenho do nosso modelo usando validação cruzada estratificada com $5$ folds.

In [9]:
def evaluate_pipeline(X, y):
    pipeline = create_pipeline(X)

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cross_validator,
        scoring={
            "roc_auc": "roc_auc",
            "accuracy": "accuracy",
            "precision": "precision",
            "recall": "recall",
            "f1": "f1",
        },
        n_jobs=-1,
    )

    return scores


In [10]:
metrics = evaluate_pipeline(X, y)

In [11]:
for metric, scores in metrics.items():
    print(f"{metric:14} = {scores.mean().round(4)}")


fit_time       = 0.4318
score_time     = 0.0593
test_roc_auc   = 0.9497
test_accuracy  = 0.9291
test_precision = 0.9555
test_recall    = 0.8734
test_f1        = 0.9126


## Análise da Feature `"sopro"`

Assim como observado nos modelos anteriores, vamos investigar o impacto da feature `"sopro"` no desempenho do modelo. A hipótese é que ela pode estar causando data leakage, pois sopro cardíaco é um achado clínico que pode ser consequência da própria patologia que estamos tentando prever.

In [12]:
print("=== Com sopro ===")
metrics_com = evaluate_pipeline(X, y)
for metric, scores in metrics_com.items():
    print(f"{metric:14} = {scores.mean().round(4)}")

print()
print("=== Sem sopro ===")
metrics_sem = evaluate_pipeline(X.drop("sopro", axis="columns"), y)
for metric, scores in metrics_sem.items():
    print(f"{metric:14} = {scores.mean().round(4)}")


=== Com sopro ===
fit_time       = 0.5889
score_time     = 0.0933
test_roc_auc   = 0.9497
test_accuracy  = 0.9291
test_precision = 0.9555
test_recall    = 0.8734
test_f1        = 0.9126

=== Sem sopro ===
fit_time       = 0.5074
score_time     = 0.0565
test_roc_auc   = 0.809
test_accuracy  = 0.7522
test_precision = 0.7676
test_recall    = 0.5964
test_f1        = 0.671


## Importância das Features por Permutação

Vamos calcular a importância de cada feature por permutação para entender a contribuição individual de cada uma no modelo. Vamos primeiramente ajustar o pipeline aos dados de treinamento.

In [13]:
from sklearn.inspection import permutation_importance

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.7,
    random_state=random_state,
    stratify=y,
)

pipeline.fit(X_train, y_train)

result = permutation_importance(
    pipeline,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=5,
    random_state=random_state,
    n_jobs=-1,
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean,
    "std": result.importances_std,
}).sort_values("importance", ascending=False)

importance


,feature,importance,std
9,sopro,0.352539,0.009215
8,b2,0.009586,0.000661
10,fc,0.005960,0.001843
3,idade,0.005927,0.001456
17,hda1_faltante,0.003148,0.000053
14,motivo1,0.003070,0.000832
5,pa_sistolica,0.001585,0.000359
15,motivo2,0.001558,0.000642
28,dias_atendimento,0.001177,0.000503
0,peso,0.001150,0.000416


Com a importância por permutação podemos confirmar ou refutar a dominância da feature `"sopro"` observada nos modelos anteriores, e identificar quais features são mais relevantes para o $\text{XGBoost}$.